# 13 — Alternative ML Targets: 1R and 2R Winners

Önceki `Meta_Label = Net_Return > 0` modeli, Validation portföyünde
orijinal Robot'u geçemedi. Bu notebook trend-following sistemine daha uygun
iki hedefi araştırır:

- **Meta_Label_1R:** İşlem en az 1R kazandırdı mı?
- **Big_Winner_Label_2R:** İşlem en az 2R kazandırdı mı?

Amaç kazanma oranını yükseltmek değil; Robot'un az sayıdaki büyük trend
işlemlerini koruyup düşük kaliteli sinyalleri elemek.

Metodoloji:

1. Model seçimi yalnızca Development purged cross-validation ile yapılır.
2. Filtre/eşik seçimi yalnızca Validation portföy backtestiyle yapılır.
3. Hiçbir aday katı kabul şartlarını sağlamazsa final sistem Baseline Robot kalır.
4. Audit dönemi yalnızca Validation kararı verildikten sonra raporlanır.
5. Audit dönemi daha önce görüldüğü için gerçek anlamda dokunulmamış test değildir.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from joblib import dump

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.features import add_indicators
from src.signals import (
    build_market_regime,
    add_robot_scores,
)
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.ml_dataset import (
    BASE_FEATURE_COLUMNS,
    add_meta_features,
)
from src.ml_training import (
    build_candidate_models,
    create_purged_expanding_folds,
    folds_summary,
    cross_validate_models,
    train_on_development_predict_validation,
)
from src.ml_portfolio import (
    MLFilterConfig,
    add_model_probabilities,
    run_filter_grid,
    add_baseline_differences,
    validation_acceptance_table,
)
from src.ml_targets import (
    add_alternative_targets,
    target_diagnostics,
    summarize_cv_lift,
    select_cv_champion,
    target_threshold_table,
    build_validation_filter_grid,
    combine_acceptance_results,
    select_global_validation_champion,
)
from src.benchmark import build_benchmark_equity
from src.metrics import portfolio_metrics


## 1. Olay veri setini ve temiz fiyatları yükle


In [ ]:
events = pd.read_parquet(
    PROJECT_ROOT
    / "results"
    / "ml"
    / "robot_meta_label_training.parquet"
)

for column in [
    "Signal_Date",
    "Entry_Date",
    "Exit_Date",
]:
    events[column] = pd.to_datetime(events[column])

events = add_alternative_targets(events)

stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

TARGET_COLUMNS = [
    "Meta_Label_1R",
    "Big_Winner_Label_2R",
]

display(
    target_diagnostics(
        dataset=events,
        target_columns=TARGET_COLUMNS,
    )
)


## 2. Günlük Robot özelliklerini hazırla


In [ ]:
stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(market_features)

scored_prices = add_robot_scores(
    stock_features=stock_features,
    market_regime=market_regime,
    config=FINAL_STRATEGY_CONFIG,
    include_reasons=False,
)

featured_prices = add_meta_features(
    scored_prices=scored_prices,
    market_features=market_features,
)

VALIDATION_START = "2023-01-01"
VALIDATION_END = "2024-12-31"
AUDIT_START = "2025-01-01"
AUDIT_END = min(
    featured_prices["Date"].max(),
    market_prices["Date"].max(),
).strftime("%Y-%m-%d")

print("Validation:", VALIDATION_START, "→", VALIDATION_END)
print("Audit:", AUDIT_START, "→", AUDIT_END)


## 3. Her hedef için Development model seçimi ve Validation portföy testi

Bu bölüm birkaç dakika sürebilir. Her hedefte:

- 4 purged expanding fold
- 3 aday model
- Validation olasılık quantile filtreleri
- Günlük Top %75 / %50 / %30 filtreleri

çalıştırılır.


In [ ]:
candidate_model_templates = build_candidate_models(
    feature_columns=BASE_FEATURE_COLUMNS,
    random_state=42,
)

target_artifacts = {}
acceptance_frames = []

for target_column in TARGET_COLUMNS:
    print("=" * 90)
    print("TARGET:", target_column)

    development = (
        events.loc[
            events["Period"].eq("Development")
        ]
        .sort_values(["Signal_Date", "Ticker"])
        .reset_index(drop=True)
    )

    validation = (
        events.loc[
            events["Period"].eq("Validation")
        ]
        .sort_values(["Signal_Date", "Ticker"])
        .reset_index(drop=True)
    )

    folds = create_purged_expanding_folds(
        development_data=development,
        n_splits=4,
        initial_train_fraction=0.40,
        embargo_days=5,
    )

    fold_table = folds_summary(
        data=development,
        folds=folds,
        target_column=target_column,
    )

    if not fold_table["Purged_Correctly"].all():
        raise RuntimeError(
            f"Purged fold kontrolü başarısız: {target_column}"
        )

    cv_fold_metrics, cv_predictions = (
        cross_validate_models(
            development_data=development,
            feature_columns=BASE_FEATURE_COLUMNS,
            target_column=target_column,
            models=candidate_model_templates,
            folds=folds,
        )
    )

    cv_lift_summary = summarize_cv_lift(
        cv_fold_metrics
    )

    cv_champion_name = select_cv_champion(
        cv_lift_summary
    )

    print("Development CV champion:", cv_champion_name)
    display(cv_lift_summary)
    display(fold_table)

    selected_template = {
        cv_champion_name: candidate_model_templates[
            cv_champion_name
        ]
    }

    (
        fitted_models,
        validation_event_metrics,
        validation_event_predictions,
    ) = train_on_development_predict_validation(
        development_data=development,
        validation_data=validation,
        feature_columns=BASE_FEATURE_COLUMNS,
        target_column=target_column,
        models=selected_template,
        cutoff_date=VALIDATION_START,
        embargo_days=5,
    )

    validation_model = fitted_models[
        cv_champion_name
    ]

    champion_event_predictions = (
        validation_event_predictions.loc[
            validation_event_predictions["Model"].eq(
                cv_champion_name
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    event_thresholds = target_threshold_table(
        predictions=champion_event_predictions,
        target_column=target_column,
    )

    validation_probability_prices = (
        add_model_probabilities(
            featured_prices=featured_prices,
            fitted_model=validation_model,
            feature_columns=BASE_FEATURE_COLUMNS,
            start=VALIDATION_START,
            end=VALIDATION_END,
        )
    )

    filter_grid, quantile_table = (
        build_validation_filter_grid(
            validation_probability_prices
        )
    )

    validation_portfolio_results, portfolio_outputs = (
        run_filter_grid(
            probability_prices=validation_probability_prices,
            filters=filter_grid,
            strategy_config=FINAL_STRATEGY_CONFIG,
            portfolio_config=FINAL_PORTFOLIO_CONFIG,
            start=VALIDATION_START,
            end=VALIDATION_END,
        )
    )

    validation_portfolio_results = (
        add_baseline_differences(
            validation_portfolio_results
        )
    )

    acceptance_table = validation_acceptance_table(
        validation_results=validation_portfolio_results,
        baseline_name="Baseline_Robot",
        minimum_trade_fraction=0.50,
        maximum_drawdown_deterioration_pp=3.0,
    )

    acceptance_table["Target"] = target_column
    acceptance_table["Model"] = cv_champion_name
    acceptance_frames.append(acceptance_table)

    print("VALIDATION EVENT METRİKLERİ")
    display(validation_event_metrics)

    print("OLASILIK QUANTILE EŞİKLERİ")
    display(quantile_table)

    print("VALIDATION PORTFÖY SONUÇLARI")
    display(
        validation_portfolio_results[
            [
                "Filter_Name",
                "CAGR_%",
                "Max_Drawdown_%",
                "Profit_Factor",
                "Sharpe",
                "Calmar",
                "Trade_Count",
                "Signal_Pass_Rate_%",
            ]
        ].sort_values(
            ["Calmar", "CAGR_%"],
            ascending=False,
        )
    )

    print("KABUL TABLOSU")
    display(
        acceptance_table[
            [
                "Target",
                "Model",
                "Filter_Name",
                "CAGR_%",
                "Max_Drawdown_%",
                "Profit_Factor",
                "Sharpe",
                "Calmar",
                "Trade_Count",
                "Acceptance_Count",
                "ML_Accepted",
            ]
        ]
    )

    target_artifacts[target_column] = {
        "development": development,
        "validation": validation,
        "fold_table": fold_table,
        "cv_fold_metrics": cv_fold_metrics,
        "cv_lift_summary": cv_lift_summary,
        "cv_champion_name": cv_champion_name,
        "validation_model": validation_model,
        "validation_event_metrics": (
            validation_event_metrics
        ),
        "validation_event_predictions": (
            champion_event_predictions
        ),
        "event_thresholds": event_thresholds,
        "validation_probability_prices": (
            validation_probability_prices
        ),
        "filter_grid": filter_grid,
        "quantile_table": quantile_table,
        "validation_portfolio_results": (
            validation_portfolio_results
        ),
        "acceptance_table": acceptance_table,
        "portfolio_outputs": portfolio_outputs,
    }


## 4. Hedefler arasında tek Validation kararı


In [ ]:
combined_acceptance = combine_acceptance_results(
    acceptance_frames
)

global_champion, ml_accepted = (
    select_global_validation_champion(
        combined_acceptance
    )
)

display(
    combined_acceptance[
        [
            "Target",
            "Model",
            "Filter_Name",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
            "Signal_Pass_Rate_%",
            "Acceptance_Count",
            "ML_Accepted",
        ]
    ].sort_values(
        [
            "ML_Accepted",
            "Calmar",
            "CAGR_%",
        ],
        ascending=[False, False, False],
    )
)

print("Alternatif hedef ML kabul edildi mi?:", ml_accepted)
print("Global Validation champion:", global_champion)


ML'nin kabul edilmesi için filtre şu koşulların tamamını sağlamalıdır:

- Baseline işlemlerin en az %50'sini korumalı.
- Validation CAGR yükselmeli.
- Validation Calmar yükselmeli.
- Profit Factor düşmemeli.
- Drawdown en fazla 3 yüzde puan kötüleşmeli.

Hiçbir aday kabul edilmezse Audit'te yalnızca Baseline Robot raporlanır.


## 5. Validation kararı sonrası Audit raporu


In [ ]:
audit_filters = [
    MLFilterConfig(
        name="Baseline_Robot",
    )
]

audit_model = None
selected_target = None
selected_model_name = None
selected_filter_name = "Baseline_Robot"
selected_filter_config = None

if ml_accepted:
    selected_target = str(
        global_champion["Target"]
    )
    selected_model_name = str(
        global_champion["Model"]
    )
    selected_filter_name = str(
        global_champion["Filter_Name"]
    )

    development_validation = (
        events.loc[
            events["Period"].isin(
                ["Development", "Validation"]
            )
        ]
        .sort_values(["Signal_Date", "Ticker"])
        .reset_index(drop=True)
    )

    audit_events = (
        events.loc[
            events["Period"].eq(
                "Audit_2025_Plus"
            )
        ]
        .sort_values(["Signal_Date", "Ticker"])
        .reset_index(drop=True)
    )

    selected_templates = {
        selected_model_name: candidate_model_templates[
            selected_model_name
        ]
    }

    (
        audit_fitted_models,
        audit_event_metrics,
        audit_event_predictions,
    ) = train_on_development_predict_validation(
        development_data=development_validation,
        validation_data=audit_events,
        feature_columns=BASE_FEATURE_COLUMNS,
        target_column=selected_target,
        models=selected_templates,
        cutoff_date=AUDIT_START,
        embargo_days=5,
    )

    audit_model = audit_fitted_models[
        selected_model_name
    ]

    selected_filter_config = next(
        config
        for config in target_artifacts[
            selected_target
        ]["filter_grid"]
        if config.name == selected_filter_name
    )

    audit_filters.append(
        selected_filter_config
    )

    display(audit_event_metrics)

if ml_accepted:
    audit_probability_prices = add_model_probabilities(
        featured_prices=featured_prices,
        fitted_model=audit_model,
        feature_columns=BASE_FEATURE_COLUMNS,
        start=AUDIT_START,
        end=AUDIT_END,
    )
else:
    audit_probability_prices = featured_prices.copy()
    audit_probability_prices[
        "ML_Probability"
    ] = np.nan

audit_results, audit_outputs = run_filter_grid(
    probability_prices=audit_probability_prices,
    filters=audit_filters,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    start=AUDIT_START,
    end=AUDIT_END,
)

audit_results = add_baseline_differences(
    audit_results
)

display(
    audit_results[
        [
            "Filter_Name",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
            "Signal_Pass_Rate_%",
        ]
    ]
)


## 6. Audit: Robot, kabul edilen ML ve BIST100


In [ ]:
audit_comparison_rows = []

for filter_name, output in audit_outputs.items():
    metrics = portfolio_metrics(
        output["equity"],
        output["trades"],
    )
    metrics["Portfolio"] = filter_name
    audit_comparison_rows.append(metrics)

baseline_equity = audit_outputs[
    "Baseline_Robot"
]["equity"]

bist100_equity = build_benchmark_equity(
    market_prices=market_prices,
    comparison_dates=baseline_equity["Date"],
    initial_capital=(
        FINAL_PORTFOLIO_CONFIG.initial_capital
    ),
    include_costs=False,
    benchmark_name="BIST100",
)

bist100_metrics = portfolio_metrics(
    bist100_equity,
    pd.DataFrame(columns=["Return"]),
)
bist100_metrics["Portfolio"] = "BIST100 Gross"
audit_comparison_rows.append(bist100_metrics)

audit_comparison = pd.DataFrame(
    audit_comparison_rows
)

display(
    audit_comparison[
        [
            "Portfolio",
            "End_Value",
            "Total_Return_%",
            "CAGR_%",
            "Max_Drawdown_%",
            "Sharpe",
            "Calmar",
            "Profit_Factor",
            "Trade_Count",
        ]
    ]
)


## 7. Sonuçları ve kararı kaydet


In [ ]:
ML_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "ml"
)
ML_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

combined_acceptance.to_csv(
    ML_RESULTS_DIR
    / "alternative_targets_validation_acceptance.csv",
    index=False,
)

audit_results.to_csv(
    ML_RESULTS_DIR
    / "alternative_targets_audit_results.csv",
    index=False,
)

audit_comparison.to_csv(
    ML_RESULTS_DIR
    / "alternative_targets_audit_vs_bist100.csv",
    index=False,
)

for target_column, artifacts in target_artifacts.items():
    safe_target = target_column.lower()

    artifacts["cv_fold_metrics"].to_csv(
        ML_RESULTS_DIR
        / f"{safe_target}_cv_fold_metrics.csv",
        index=False,
    )

    artifacts["cv_lift_summary"].to_csv(
        ML_RESULTS_DIR
        / f"{safe_target}_cv_lift_summary.csv",
        index=False,
    )

    artifacts[
        "validation_event_metrics"
    ].to_csv(
        ML_RESULTS_DIR
        / f"{safe_target}_validation_event_metrics.csv",
        index=False,
    )

    artifacts["event_thresholds"].to_csv(
        ML_RESULTS_DIR
        / f"{safe_target}_event_thresholds.csv",
        index=False,
    )

    artifacts[
        "validation_portfolio_results"
    ].to_csv(
        ML_RESULTS_DIR
        / f"{safe_target}_validation_portfolio.csv",
        index=False,
    )

decision = {
    "alternative_targets_tested": TARGET_COLUMNS,
    "ml_accepted_on_validation": bool(ml_accepted),
    "selected_target": selected_target,
    "selected_model": selected_model_name,
    "selected_filter": selected_filter_name,
    "final_system": (
        "Robot_with_alternative_target_ML"
        if ml_accepted
        else "Baseline_Robot"
    ),
    "validation_period": {
        "start": VALIDATION_START,
        "end": VALIDATION_END,
    },
    "audit_period": {
        "start": AUDIT_START,
        "end": AUDIT_END,
    },
}

DECISION_PATH = (
    MODEL_DIR
    / "alternative_target_ml_decision.json"
)

with DECISION_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision,
        file,
        ensure_ascii=False,
        indent=2,
    )

if ml_accepted and audit_model is not None:
    dump(
        audit_model,
        MODEL_DIR
        / "alternative_target_ml_model.joblib",
    )

print("Karar:", decision)
print("Karar dosyası:", DECISION_PATH)


## Karar ilkesi

- Validation'da hiçbir alternatif hedef kabul edilmezse ML araştırması burada
  durdurulur ve günlük sistem Baseline Robot olarak devam eder.
- Validation'da kabul edilip Audit'te belirgin biçimde bozulursa ML canlı
  sisteme alınmaz.
- Validation ve Audit olumlu olsa bile gerçek yeni out-of-sample doğrulama
  paper trading kayıtlarıdır.
